# F&B Sales Analytics: аналіз продажів мережі кав'ярень

**Індонезія · 3 торгові точки · січень–вересень 2025**

## Про проєкт
Мережа кав'ярень у `SHOP001`, `SHOP002` і `SHOP003` зберігає кожен продаж у касовій системі (POS). У цьому ноутбуці я аналізую ці дані: **53 820 чеків, 61 973 продані позиції, 96 товарів у меню**. Валюта: індонезійська рупія (IDR).

Мета: зрозуміти, **де, коли і на чому** бізнес заробляє, і запропонувати кілька ідей для зростання.

## На які питання відповідає аналіз
1. Скільки заробляє мережа і який середній чек?
2. Чим відрізняються три заклади?
3. Як змінювалися продажі по місяцях і що сталося в квітні?
4. У які дні й години найбільше покупців?
5. Які категорії та продукти приносять найбільше доходу?
6. Як працює програма лояльності?
7. Як клієнти платять?
8. Що купують разом в одному чеку?

## Дані
| Файл | Що в ньому |
|---|---|
| `Transactions.csv` | один рядок = один чек (заклад, дата, час, сума, список товарів, спосіб оплати) |
| `Products.csv` | довідник меню (назва, варіант, категорія, ціна) |

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


BLUE = '#1F4E79'
LIGHT_BLUE = '#8FB8DE'
ORANGE = '#E8742A'
GREY = '#9A9A9A'
GREEN = '#2E8B57'
RED = '#C0392B'

# кожен заклад має свій колір у всіх графіках
OUTLET_COLORS = {'SHOP001': BLUE, 'SHOP002': ORANGE, 'SHOP003': LIGHT_BLUE}

In [2]:
# шукаю папку з файлами: поруч із ноутбуком або в ../data/
DATA_DIR = None
for folder in ['../data/', 'data/', '']:
    if os.path.exists(folder + 'Transactions.csv'):
        DATA_DIR = folder
        break
if DATA_DIR is None:
    raise FileNotFoundError('Не знайшов Transactions.csv. Покладіть файлиф в папку data/ або поруч із ноутбуком.')

products = pd.read_csv(DATA_DIR + 'Products.csv')
transactions = pd.read_csv(DATA_DIR + 'Transactions.csv')

print(f'Products: {products.shape[0]} рядків, {products.shape[1]} колонок')
print(f'Transactions: {transactions.shape[0]} рядків, {transactions.shape[1]} колонок')
display(products.head(3))
display(transactions.head(3))

Products: 96 рядків, 6 колонок
Transactions: 53820 рядків, 11 колонок


,ProductId,ProductName,Variant,Category,Price,Description
0,1,Add on Aren,-,Add on,0,"Tambahan sirup gula aren murni, memberikan ras..."
1,2,Add on Simple Syrup,-,Add on,0,Tambahan sirup gula sederhana (air dan gula) s...
2,3,Air Mineral,-,Other,5000,Air minum murni dalam kemasan


,Outlet,Date,Time,NetSales,Tax,TotalAmount,ReceiptNumber,Items,TotalItem,PaymentMethod,UseLoyaltyCard
0,SHOP001,2025-01-02,07:01:05,20720.721,2279.279,23000,SHOP001JAN2500001,Basic Latte (Ice Arabica),1,BCA,0
1,SHOP001,2025-01-02,07:04:11,18018.018,1981.982,20000,SHOP001JAN2500002,BLACK (Houseblend),1,BCA,0
2,SHOP002,2025-01-02,07:10:46,39639.640,4360.360,44000,SHOP002JAN2500001,"Friendly Coffee (Hot), Shakencano",2,Cash,0


In [3]:
# перетворюю дату й час та додаю зручні колонки
transactions['Date'] = pd.to_datetime(transactions['Date'])
transactions['Month'] = transactions['Date'].dt.to_period('M').astype(str)
transactions['Weekday_num'] = transactions['Date'].dt.dayofweek          # 0 = понеділок
transactions['Hour'] = pd.to_datetime(transactions['Time'], format='%H:%M:%S').dt.hour

# перевірки якості
print('Пропуски в Products:', products.isna().sum().sum())
print('Пропуски в Transactions:', transactions.isna().sum().sum())
print('Дублікати номерів чеків:', transactions['ReceiptNumber'].duplicated().sum())
print('Від\'ємні суми чеків:', (transactions['TotalAmount'] < 0).sum())
print('Чеки з нульовою сумою:', (transactions['TotalAmount'] == 0).sum())
print('Заклади:', sorted(transactions['Outlet'].unique()))
print('Способи оплати:', sorted(transactions['PaymentMethod'].unique()))
print('Період:', transactions['Date'].min().date(), '—', transactions['Date'].max().date())

Пропуски в Products: 0
Пропуски в Transactions: 0
Дублікати номерів чеків: 0
Від'ємні суми чеків: 0
Чеки з нульовою сумою: 1733
Заклади: ['SHOP001', 'SHOP002', 'SHOP003']
Способи оплати: ['BCA', 'Cash', 'GoFood']
Період: 2025-01-02 — 2025-09-30


C:\Users\Дмитро\AppData\Local\Temp\ipykernel_25596\863099258.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  transactions['Weekday_num'] = transactions['Date'].dt.dayofweek          # 0 = понеділок


 Чеки з **нульовою сумою** не помилка: це чеки, оплачені програмою лояльності (клієнт отримує позицію безкоштовно). Їх я розберу в окремому розділі.

## 3. Підготовка таблиці проданих позицій
У чеку всі товари записані в одному полі `Items` через кому, наприклад `BLACK (Arabica), Americano (Hot Arabica)`. Щоб рахувати продажі по продуктах і категоріях, я:
1. складаю повну назву товару в довіднику (`Назва (Варіант)`),
2. розділяю список товарів у чеку на окремі рядки (`explode`),
3. приєдную довідник, щоб отримати категорію та ціну.

In [4]:
# повна назва товару, як вона записана в чеках
products['FullName'] = np.where(
    products['Variant'] == '-',
    products['ProductName'],
    products['ProductName'] + ' (' + products['Variant'] + ')'
)

# один рядок = одна позиція в чеку
items_df = transactions[['ReceiptNumber', 'Outlet', 'Month', 'PaymentMethod', 'UseLoyaltyCard', 'TotalItem', 'Items']].copy()
items_df['Item'] = items_df['Items'].str.split(', ')
items_df = items_df.explode('Item')

# приєдную категорію та ціну з довідника
items_df = items_df.merge(
    products[['ProductId', 'FullName', 'ProductName', 'Category', 'Price']],
    left_on='Item', right_on='FullName', how='left'
)

# якщо клієнт платив карткою лояльності, позиція безкоштовна
items_df['Revenue'] = np.where(items_df['UseLoyaltyCard'] == 1, 0, items_df['Price'])
items_df['Loyalty_Discount'] = np.where(items_df['UseLoyaltyCard'] == 1, items_df['Price'], 0)

print(f'Позицій у таблиці: {len(items_df):,} (сума TotalItem у чеках: {transactions["TotalItem"].sum():,})')
print(f'Товарів, яких не знайшли в довіднику: {items_df["ProductId"].isna().sum()}')

Позицій у таблиці: 61,973 (сума TotalItem у чеках: 61,973)
Товарів, яких не знайшли в довіднику: 0
